In [1]:
%pip install mcp python-dotenv
%pip install langchain-ollama


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [5]:
# Test to see if the local llm is running and reachable.
from langchain_ollama import ChatOllama
local_model="qwen3:0.6b"
local_llm = ChatOllama(
    model = local_model, 
    # Dont' make things up.
    temperature = 0,
    stream = True,
)

# Make sure the model name matches exactly what you see in `ollama list`
for chunk in local_llm.stream("How is the weather in Tehran right now?"):
    print(chunk.content, end="", flush=True)

The current weather in Tehran is characterized by a dry and hot climate, with temperatures typically ranging from 40°C to 50°C. The air is dry, and there may be a slight breeze. It's important to note that Tehran experiences a continental climate, which makes it quite hot and dry throughout the day.

In [6]:
# Now lets start the weather mcp server.

import asyncio
import ollama
import json
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client

# 1. Define your MCP Server (e.g., the Weather or News server)
server_params = StdioServerParameters(
    command="npx", 
    args=["-y", "@newsmcp/server"], # Example: News/Search
    env=None
)

async def run_ollama_mcp():
    async with stdio_client(server_params) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            
            # 2. Get tools from MCP and format them for Ollama
            mcp_tools = await session.list_tools()
            ollama_tools = [
                {
                    "type": "function",
                    "function": {
                        "name": t.name,
                        "description": t.description,
                        "parameters": t.inputSchema,
                    },
                }
                for t in mcp_tools.tools
            ]
            print(f"Ollama tools:{json.dumps(ollama_tools,indent=2)}")
            print("---Done printing tools")
            # 3. Chat with Ollama
            messages = [
                {'role': 'system',
                 'content' : '''
                 You are a news curator. 
                  CRITICAL RULE: Always set the 'per_page' parameter to strictly 3
                  when calling the 'get_news' tool to save memory. 
                 
                 When presenting news:
                    1. Extract only the Title and a 1-sentence summary.
                    2. DO NOT include URLs, IDs, Impact scores, or Regions.
                    3. Use a clean bulleted list format."
                 You have two tools
                    1. `get_news`: Use this first to get a list of headlines and their UUIDs.
                    2. `get_news_detail`: Once you have a UUID from the list, you MUST call this tool to get the full story.

                    **Instructions:**
                    - When the user asks for news, call `get_news`.
                    - Look at the response. Find the UUID for the relevant article.
                    - Immediately call `get_news_detail` using that UUID.
                    - Do not ask the user for permission; just get the details and then summarize the final result.
                    
                 '''},
                {"role": "user", "content": "Get met latest news from the USA?"}]
            response = ollama.chat(
                model=local_model, # Make sure this model supports tools!
                messages=messages,
                tools=ollama_tools,
            )

            # 4. Handle Tool Calls
            if response.get('message', {}).get('tool_calls'):
                for tool_call in response['message']['tool_calls']:
                    # Execute the tool via the MCP Session
                    result = await session.call_tool(
                        tool_call['function']['name'], 
                        tool_call['function']['arguments']
                    )
                    
                    # Format the MCP result into an Ollama tool response message
                    # We extract the text content from the MCP response
                    tool_output_text = " ".join([c.text for c in result.content if hasattr(c, 'text')])
        
                    messages.append({
                        'role': 'tool',
                        'content': tool_output_text,
                    })
            # 5. FINAL STEP: Send the tool results back to Ollama for the clean summary
            final_response = ollama.chat(
                model=local_model,
                messages=messages,
            )
            print("\n--- FINAL CLEAN SUMMARY ---")
            print(final_response['message']['content'])

# Run in Jupyter
await run_ollama_mcp()



Ollama tools:[
  {
    "type": "function",
    "function": {
      "name": "get_news",
      "description": "Get top news events happening in the world right now. Returns AI-clustered, deduplicated news stories ranked by importance. Present results as a multi-story news briefing \u2014 cover the top events, not just one. Each event should be 1-2 lines with its summary and 1-2 source links. Prioritize suppressing rich link cards/previews over pretty link labels. For Discord-style clients, output source URLs directly as '<https://...>' and do not use masked markdown links. Never emit raw standalone URLs outside no-preview wrappers. Only deep-dive into a specific event if the user asks for detail.",
      "parameters": {
        "type": "object",
        "properties": {
          "topics": {
            "type": "string",
            "description": "Comma-separated topic slugs (e.g. 'politics,technology'). Use get_topics to see available topics."
          },
          "geo": {
           